# Solving CartPole-v1 with PPO (from scratch)

This notebook demonstrates how to solve the CartPole-v1 environment using a custom implementation of the Proximal Policy Optimization (PPO) algorithm in PyTorch. No external RL libraries are used.

---

**Note:** All environment and hyperparameter settings are now collected in a single `CONFIG` dictionary at the top of the notebook. To change the environment or any hyperparameter, simply edit the values in the config cell.

## 1. Install and Import Required Libraries
We will use gymnasium and torch for this implementation.

In [ ]:
%pip install gymnasium stable-baselines3 wandb tsilva-notebook-utils==0.0.121 --quiet

In [ ]:
from tsilva_notebook_utils.colab import load_secrets_into_env

_ = load_secrets_into_env([
    'WANDB_API_KEY'
])

In [ ]:
from tsilva_notebook_utils.torch import get_default_device
DEVICE = get_default_device()
DEVICE

## 2. Set Up CartPole-v1 Environment
We will initialize the CartPole-v1 environment and display its basic information.

In [ ]:
import torch.nn as nn
from tsilva_notebook_utils.gymnasium import build_env as _build_env, set_random_seed
from dataclasses import dataclass
from typing import Union, Tuple

@dataclass
class PPOConfig:
    # Environment
    env_id: str = "CartPole-v1"
    seed: int = 42
    
    # Training
    max_epochs: int = -1
    gamma: float = 0.99
    lam: float = 0.95
    clip_epsilon: float = 0.2
    minibatch_size: int = 64
    train_rollout_steps: int = 2048
    
    # Evaluation
    eval_interval: int = 10
    eval_episodes: int = 32
    reward_threshold: float = 200
    
    # Networks
    policy_lr: float = 3e-4
    value_lr: float = 1e-3
    hidden_dim: Union[int, Tuple[int, ...]] = 64
    entropy_coef: float = 0.01
    
    # Other
    normalize: bool = False
    mean_reward_window: int = 100
    rollout_interval: int = 10
    n_envs: Union[str, int] = "auto"
    async_rollouts: bool = True
    
    @classmethod
    def for_env(cls, env_id: str) -> 'PPOConfig':
        """Factory method for environment-specific configs"""
        base = cls(env_id=env_id)
        
        env_overrides = {
            "CartPole-v1": dict(
                train_rollout_steps=512,
                minibatch_size=256,
                rollout_interval=1,
                eval_interval=20,
                eval_episodes=5,
                reward_threshold=475,
                policy_lr=1e-3,
                value_lr=1e-3,
                hidden_dim=32,
            ),
            "LunarLander-v3": dict(
                gamma=0.99,
                lam=0.95,
                clip_epsilon=0.2,
                minibatch_size=64,
                eval_interval=2,
                reward_threshold=200,
                policy_lr=1e-4,
                value_lr=5e-4,
                hidden_dim=32,
                entropy_coef=0.02
            ),
            "Acrobot-v1": dict(
                gamma=0.99,
                lam=0.95,
                clip_epsilon=0.2,
                minibatch_size=32,
                eval_interval=2,
                reward_threshold=-100,
                policy_lr=3e-4,
                value_lr=1e-3,
                hidden_dim=64,
                entropy_coef=0.01
            ),
            "Pendulum-v1": dict(
                gamma=0.99,
                lam=0.95,
                clip_epsilon=0.2,
                minibatch_size=64,
                eval_interval=2,
                eval_episodes=5,
                reward_threshold=-200,
                policy_lr=3e-4,
                value_lr=1e-3,
                hidden_dim=(128, 64),
                entropy_coef=0.0
            ),
            "MountainCar-v0": dict(
                gamma=0.99,
                lam=0.97,
                clip_epsilon=0.15,
                minibatch_size=16,
                eval_interval=2,
                eval_episodes=10,
                reward_threshold=-110,
                policy_lr=1e-4,
                value_lr=5e-4,
                hidden_dim=(128, 64),
                entropy_coef=0.05
            ),
        }
        
        if env_id in env_overrides:
            for key, value in env_overrides[env_id].items():
                setattr(base, key, value)
        
        return base

ENV_ID = "CartPole-v1"
#ENV_ID = "Acrobot-v1"
#ENV_ID = "LunarLander-v3"
#ENV_ID = "Pendulum-v1"
#ENV_ID = "MountainCar-v0"
CONFIG = PPOConfig.for_env(ENV_ID)
CONFIG

In [ ]:
from tsilva_notebook_utils.gymnasium import log_env_info

# Set random seed for reproducibility
set_random_seed(CONFIG.seed)

# Wrap build env with config parameters
build_env = lambda seed, n_envs=None: _build_env(
    CONFIG.env_id, 
    norm_obs=CONFIG.normalize, 
    n_envs=n_envs if n_envs is not None else CONFIG.n_envs, 
    seed=seed
)

# Test building env
env = build_env(CONFIG.seed)
log_env_info(env)

## 3. Implement PPO Agent
We will define the policy and value networks, and the PPO update step.

In [ ]:
class MLPNet(nn.Module):
    """Reusable MLP with configurable hidden dimensions"""
    
    def __init__(self, input_dim, output_dim, hidden_dim=64, activation=nn.ReLU):
        super().__init__()
        
        if isinstance(hidden_dim, (int, float)):
            hidden_dims = [int(hidden_dim)]
        else:
            hidden_dims = [int(dim) for dim in hidden_dim]
        
        layers = []
        current_dim = input_dim
        
        for hidden_size in hidden_dims:
            layers.extend([
                nn.Linear(current_dim, hidden_size),
                activation()
            ])
            current_dim = hidden_size
        
        layers.append(nn.Linear(current_dim, output_dim))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

class PolicyNet(MLPNet):
    def __init__(self, obs_dim, act_dim, hidden_dim=64):
        super().__init__(obs_dim, act_dim, hidden_dim)

class ValueNet(MLPNet):
    def __init__(self, obs_dim, hidden_dim=64):
        super().__init__(obs_dim, 1, hidden_dim)

In [ ]:

import torch
from torch.distributions import Categorical

class PPOLoss:
    def __init__(self, clip_epsilon, entropy_coef):
        self.clip_epsilon = clip_epsilon
        self.entropy_coef = entropy_coef
    
    def compute(self, states, actions, old_logps, advantages, returns, policy_model, value_model):
        # Policy loss
        logits = policy_model(states)
        dist = Categorical(logits=logits)
        new_logps = dist.log_prob(actions)
        
        ratio = torch.exp(new_logps - old_logps)
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1.0 - self.clip_epsilon, 1.0 + self.clip_epsilon) * advantages
        entropy = dist.entropy().mean()
        
        policy_loss = -torch.min(surr1, surr2).mean() - self.entropy_coef * entropy
        
        # Value loss
        value_pred = value_model(states).squeeze()
        value_loss = 0.5 * ((returns - value_pred) ** 2).mean()
        
        # Metrics
        clip_fraction = ((ratio < 1.0 - self.clip_epsilon) | (ratio > 1.0 + self.clip_epsilon)).float().mean()
        kl_div = (old_logps - new_logps).mean()
        approx_kl = ((ratio - 1) - torch.log(ratio)).mean()
        explained_var = 1 - torch.var(returns - value_pred) / torch.var(returns)
        
        return {
            'policy_loss': policy_loss,
            'value_loss': value_loss,
            'entropy': entropy,
            'clip_fraction': clip_fraction,
            'kl_div': kl_div,
            'approx_kl': approx_kl,
            'explained_var': explained_var
        }


In [ ]:
class MetricTracker:
    """Unified metric collection and logging system"""
    
    def __init__(self, logger=None):
        self.logger = logger
        self.reset()
    
    def reset(self):
        """Reset epoch-level metrics"""
        self.step_metrics = []
    
    def add_step_metrics(self, metrics_dict):
        """Add metrics from a single training step"""
        self.step_metrics.append({k: v.detach() if hasattr(v, 'detach') else v 
                                 for k, v in metrics_dict.items()})
    
    def compute_epoch_means(self):
        """Compute mean of all step metrics for the epoch"""
        if not self.step_metrics:
            return {}
        
        epoch_metrics = {}
        for key in self.step_metrics[0].keys():
            values = [m[key] for m in self.step_metrics]
            epoch_metrics[key] = torch.stack(values).mean() if hasattr(values[0], 'dim') else np.mean(values)
        
        return epoch_metrics
    
    def log_metrics(self, metrics_dict, prefix="", prog_bar=False):
        """Log metrics with optional prefix"""
        if not self.logger:
            return
            
        formatted_metrics = {}
        for key, value in metrics_dict.items():
            if value is not None:
                full_key = f"{prefix}/{key}" if prefix else key
                formatted_metrics[full_key] = value
        
        if formatted_metrics:
            self.logger.log_dict(formatted_metrics, prog_bar=prog_bar)
    
    def log_single(self, key, value, prog_bar=False):
        """Log a single metric"""
        if self.logger and value is not None:
            self.logger.log(key, value, prog_bar=prog_bar)

In [ ]:
import time
import torch
import multiprocessing
import numpy as np
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from collections import deque
import threading
import queue
import copy
from tsilva_notebook_utils.gymnasium import RolloutDataset, collect_rollouts, group_trajectories_by_episode

class BaseRolloutCollector:
    """Base class for rollout collectors"""
    def __init__(self, build_env_fn, config: PPOConfig, obs_dim, act_dim):
        self.build_env_fn = build_env_fn
        self.config = config
        self.obs_dim = obs_dim
        self.act_dim = act_dim
        self.last_obs = None
        
    def start(self):
        """Start the collector"""
        pass
        
    def stop(self):
        """Stop the collector"""
        pass
        
    def update_models(self, policy_state_dict, value_state_dict):
        """Update model weights"""
        pass
        
    def get_rollout(self, timeout=1.0):
        """Get next rollout data"""
        raise NotImplementedError
        
    def initialize_with_models(self, policy_model, value_model):
        """Initialize collector with model references"""
        pass

class SyncRolloutCollector(BaseRolloutCollector):
    """Synchronous rollout collector - collects data on demand"""
    def __init__(self, build_env_fn, config: PPOConfig, obs_dim, act_dim):
        super().__init__(build_env_fn, config, obs_dim, act_dim)
        self.env = build_env_fn(config.seed)
        self.policy_model = None
        self.value_model = None
        self._ready_for_initial = False
        
    def initialize_with_models(self, policy_model, value_model):
        """Set model references for sync collector"""
        self.policy_model = policy_model
        self.value_model = value_model
        self._ready_for_initial = True
        
    def get_rollout(self, timeout=1.0):
        """Collect rollout synchronously using current models"""
        if self.policy_model is None or self.value_model is None:
            return None
            
        trajectories, extras = collect_rollouts(
            self.env,
            self.policy_model,
            self.value_model,
            n_steps=self.config.train_rollout_steps,
            last_obs=self.last_obs
        )
        
        self.last_obs = extras['last_obs']
        return trajectories
        
    def is_ready_for_initial_rollout(self):
        """Check if ready for initial rollout collection"""
        return self._ready_for_initial

class AsyncRolloutCollector(BaseRolloutCollector):
    """Background thread that continuously collects rollouts using latest model weights"""
    def __init__(self, build_env_fn, config: PPOConfig, obs_dim, act_dim):
        super().__init__(build_env_fn, config, obs_dim, act_dim)
        
        # Thread-safe queue for rollout data
        self.rollout_queue = queue.Queue(maxsize=3)  # Buffer 3 rollouts max
        
        # Shared model weights (CPU copies for thread safety)
        self.policy_state_dict = None
        self.value_state_dict = None
        self.model_lock = threading.Lock()
        
        # Control flags
        self.running = False
        self.thread = None
        
        # Create environment and models for rollout collection
        self.env = None
        self.policy_model = None
        self.value_model = None
        
    def initialize_with_models(self, policy_model, value_model):
        """Initialize with model state dicts for async collector"""
        self.update_models(policy_model.state_dict(), value_model.state_dict())
        
    def start(self):
        """Start the background rollout collection thread"""
        if self.running:
            return
            
        self.running = True
        self.thread = threading.Thread(target=self._collect_loop, daemon=True)
        self.thread.start()
        
    def stop(self):
        """Stop the background rollout collection"""
        self.running = False
        if self.thread:
            self.thread.join(timeout=5.0)
            
    def update_models(self, policy_state_dict, value_state_dict):
        """Update model weights from main training thread"""
        with self.model_lock:
            self.policy_state_dict = copy.deepcopy(policy_state_dict)
            self.value_state_dict = copy.deepcopy(value_state_dict)
            
    def get_rollout(self, timeout=1.0):
        """Get next rollout data (non-blocking with timeout)"""
        try:
            return self.rollout_queue.get(timeout=timeout)
        except queue.Empty:
            return None
            
    def is_ready_for_initial_rollout(self):
        """Check if ready for initial rollout collection"""
        return self.policy_state_dict is not None and self.value_state_dict is not None

    # TODO: called how many times?
    def _init_models(self):
        """Initialize models in the worker thread"""
        if self.env is None:
            self.env = self.build_env_fn(self.config.seed + 1000)  # Different seed for rollout env
        
        # TODO: create from outside?
        if self.policy_model is None:
            self.policy_model = PolicyNet(self.obs_dim, self.act_dim, self.config.hidden_dim)
            self.policy_model.eval()  # Always in eval mode for rollouts
            
        # TODO: create from outside?
        if self.value_model is None:
            self.value_model = ValueNet(self.obs_dim, self.config.hidden_dim)
            self.value_model.eval()
            
    def _update_model_weights(self):
        """Update local model weights from shared state dicts"""
        with self.model_lock:
            if self.policy_state_dict is not None:
                self.policy_model.load_state_dict(self.policy_state_dict)
            if self.value_state_dict is not None:
                self.value_model.load_state_dict(self.value_state_dict)
                
    def _collect_loop(self):
        """Main loop running in background thread"""
        self._init_models()
        
        while self.running:
            try:
                # Update to latest model weights
                self._update_model_weights()
                
                # Collect rollout
                trajectories, extras = collect_rollouts(
                    self.env,
                    self.policy_model,
                    self.value_model,
                    n_steps=self.config.train_rollout_steps,
                    last_obs=self.last_obs
                )
                
                self.last_obs = extras['last_obs']
                
                # Put rollout in queue (non-blocking, drop if full)
                try:
                    self.rollout_queue.put(trajectories, block=False)
                except queue.Full:
                    # Queue is full, drop oldest and add new
                    try:
                        self.rollout_queue.get_nowait()
                        self.rollout_queue.put(trajectories, block=False)
                    except queue.Empty:
                        pass
                        
            except Exception as e:
                print(f"Error in rollout collection: {e}")
                time.sleep(0.1)  # Brief pause on error


In [ ]:

# ---------------------------------------------------------------------
#  PPO Lightning module with fully unified rollout collection
# ---------------------------------------------------------------------
class PPOAgent(pl.LightningModule):
    def __init__(self, obs_dim, act_dim, config: PPOConfig):
        super().__init__()
        self.save_hyperparameters()

        # Configuration
        self.config = config
        self.obs_dim, self.act_dim = obs_dim, act_dim

        # Models
        self.policy_model = PolicyNet(obs_dim, act_dim, config.hidden_dim)
        self.value_model = ValueNet(obs_dim, config.hidden_dim)
        self.env = build_env(config.seed)

        # PPO components
        self.ppo_loss = PPOLoss(config.clip_epsilon, config.entropy_coef)
        
        # Metrics and data tracking
        self.metrics = MetricTracker(self)
        self.rollout_ds = RolloutDataset()
        self.episode_reward_deque = deque(maxlen=config.mean_reward_window)
        
        # Rollout collection
        rollout_collector_cls = AsyncRolloutCollector if config.async_rollouts else SyncRolloutCollector
        self.rollout_collector = rollout_collector_cls(build_env, config, obs_dim, act_dim)
        
        # Training state
        self.automatic_optimization = False
        self.training_start_time = None

    def setup(self, stage: str):
        if stage == "fit":
            self.rollout_collector.initialize_with_models(self.policy_model, self.value_model)
            self.rollout_collector.start()
            
            print("Waiting for initial rollout...")
            while True:
                if self.rollout_collector.is_ready_for_initial_rollout():
                    trajectories = self.rollout_collector.get_rollout(timeout=2.0)
                    if trajectories is not None:
                        self._update_rollout_data(trajectories)
                        break
                print("Still waiting for rollout...")

    def train_dataloader(self):
        return DataLoader(
            self.rollout_ds,
            batch_size=self.config.minibatch_size,
            shuffle=True,
            pin_memory=True if self.device.type != 'mps' else False,
            num_workers=multiprocessing.cpu_count() // 2 if self.device.type != 'mps' else 0
        )

    def on_fit_start(self):
        self.training_start_time = time.time()
        mode = "async" if self.config.async_rollouts else "sync"
        print(f"PPO training started in {mode} mode at {time.strftime('%Y-%m-%d %H:%M:%S')}")
    
    def on_fit_end(self):
        self.rollout_collector.stop()
        if self.training_start_time:
            total_time = time.time() - self.training_start_time
            print(f"PPO training completed in {total_time:.2f} seconds ({total_time/60:.2f} minutes)")

    def on_train_epoch_start(self):
        self.metrics.reset()
        self.rollout_collector.update_models(
            self.policy_model.state_dict(), self.value_model.state_dict()
        )
        
        # Collect new rollout if needed
        if (self.current_epoch + 1) % self.config.rollout_interval == 0:
            self._collect_and_update_rollout()

    def on_train_epoch_end(self):
        # Log epoch metrics
        epoch_metrics = self.metrics.compute_epoch_means()
        if epoch_metrics:
            self.metrics.log_metrics(epoch_metrics, prefix="epoch")
        
        # Evaluation
        if (self.current_epoch + 1) % self.config.eval_interval == 0:
            self._evaluate_and_check_stopping()

    def training_step(self, batch, batch_idx):
        opt_policy, opt_value = self.optimizers()
        states, actions, rewards, dones, old_logps, values, advantages, returns, frames = batch

        # Compute losses and metrics
        loss_results = self.ppo_loss.compute(
            states, actions, old_logps, advantages, returns, 
            self.policy_model, self.value_model
        )

        # Track step metrics
        self.metrics.add_step_metrics(loss_results)

        # Optimize
        self._optimize_models(opt_policy, opt_value, loss_results['policy_loss'], loss_results['value_loss'])

        # Log training metrics
        mean_reward = np.mean(self.episode_reward_deque) if len(self.episode_reward_deque) >= self.config.mean_reward_window else 0
        
        train_metrics = {
            'mean_reward': mean_reward,
            'policy_loss': loss_results['policy_loss'],
            'value_loss': loss_results['value_loss'],
            'entropy': loss_results['entropy'],
            'kl_divergence': loss_results['kl_div'],
            'explained_variance': loss_results['explained_var']
        }
        
        additional_metrics = {
            'approx_kl': loss_results['approx_kl'],
            'clip_fraction': loss_results['clip_fraction'],
            'advantage_mean': advantages.mean(),
            'advantage_std': advantages.std(),
            'value_mean': values.mean(),
            'returns_mean': returns.mean(),
        }
        
        self.metrics.log_metrics(train_metrics, prefix="train", prog_bar=True)
        self.metrics.log_metrics(additional_metrics, prefix="train", prog_bar=False)

        return loss_results['policy_loss'] + loss_results['value_loss']

    def configure_optimizers(self):
        return [
            torch.optim.Adam(self.policy_model.parameters(), lr=self.config.policy_lr),
            torch.optim.Adam(self.value_model.parameters(), lr=self.config.value_lr)
        ]

    def forward(self, x):
        return self.policy_model(x)

    def _collect_and_update_rollout(self):
        """Collect and update rollout data"""
        timeout = 2.0 if self.config.async_rollouts else 1.0
        trajectories = self.rollout_collector.get_rollout(timeout=timeout)
        
        if trajectories is not None:
            self._update_rollout_data(trajectories)
            self.metrics.log_single('rollout/queue_updated', 1.0)
        else:
            self.metrics.log_single('rollout/queue_miss', 1.0)
    
    def _update_rollout_data(self, trajectories):
        """Update rollout dataset and episode rewards"""
        self.rollout_ds.update(*trajectories)
        episodes = group_trajectories_by_episode(trajectories)
        episode_rewards = [sum(step[2] for step in episode) for episode in episodes]
        for r in episode_rewards:
            self.episode_reward_deque.append(float(r))

    def _optimize_models(self, opt_policy, opt_value, policy_loss, value_loss):
        """Optimize policy and value models"""
        opt_policy.zero_grad()
        self.manual_backward(policy_loss)
        opt_policy.step()

        opt_value.zero_grad()
        self.manual_backward(value_loss)
        opt_value.step()

    def _evaluate_and_check_stopping(self):
        """Evaluate model and check for early stopping"""
        eval_seed = np.random.randint(0, 1_000_000)
        eval_env = build_env(eval_seed)
        
        self.policy_model.eval()
        try:
            eval_mean_reward = self._run_evaluation(eval_env)
            self.metrics.log_single('eval/mean_reward', eval_mean_reward, prog_bar=True)
            
            if eval_mean_reward >= self.config.reward_threshold:
                print(f"Early stopping at epoch {self.current_epoch} with eval mean reward {eval_mean_reward:.2f} >= threshold {self.config.reward_threshold}")
                self.trainer.should_stop = True
                
        finally:
            self.policy_model.train()
            eval_env.close()

    def _run_evaluation(self, env):
        """Run evaluation and log rollout metrics"""
        start = time.time()
        trajectories, _ = collect_rollouts(
            env, self.policy_model, self.value_model,
            n_episodes=self.config.eval_episodes, deterministic=False
        )
        elapsed = time.time() - start

        episodes = group_trajectories_by_episode(trajectories)
        episode_rewards = [sum(step[2] for step in episode) for episode in episodes]
        mean_episode_reward = np.mean(episode_rewards)

        # Log rollout metrics
        rollout_metrics = {
            'mean_reward': mean_episode_reward,
            'num_episodes': len(episodes),
            'num_steps': len(trajectories[0]),
            'avg_steps_per_episode': len(trajectories[0]) / (len(episodes) + 1e-3),
            'time_elapsed': elapsed,
            'steps_per_second': len(trajectories[0]) / (elapsed + 1e-3)
        }
        
        self.metrics.log_metrics(rollout_metrics, prefix="rollout")
        return mean_episode_reward

## 4. Train PPO Agent
We will train the PPO agent on CartPole-v1.

In [ ]:
from pytorch_lightning import Trainer
from pytorch_lightning.loggers import WandbLogger

# Create PPO agent and move to device
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.n if hasattr(env.action_space, 'n') else env.action_space.shape[0]  # Handle discrete and continuous actions
ppo_agent = PPOAgent(obs_dim, act_dim, CONFIG)

# Set up trainer with proper device configuration
wandb_logger = WandbLogger(project="gymnasium_ppo") # TODO: softcode this

trainer = Trainer(
    logger=wandb_logger,
    log_every_n_steps=10,
    max_epochs=CONFIG.max_epochs,
    enable_progress_bar=True,
    enable_checkpointing=False,  # Disable checkpointing for speed
    accelerator="auto"
)

# Fit the model
trainer.fit(ppo_agent)

In [ ]:
import random
from tsilva_notebook_utils.gymnasium import render_episode_frames

n_episodes = 8
trajectories, _ = collect_rollouts(
    build_env(random.randint(0, 1_000_000), n_envs=n_episodes),
    ppo_agent.policy_model,
    n_episodes=n_episodes,
    deterministic=True,
    collect_frames=True
)
episodes = group_trajectories_by_episode(trajectories) # something is wrong in frame collection
mean_reward = np.mean([sum(step[2] for step in episode) for episode in episodes])
episode_frames = [[step[-1] for step in episode] for episode in episodes]
print(f"Mean reward: {mean_reward:.2f}")
render_episode_frames(episode_frames, out_dir="./tmp", grid=(2, 2), text_color=(0, 0, 0))